In [ ]:
# Import pytorch
import torch
from torch import nn
import random
# Import torch vision
import torchvision
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor

# Import matplotlib
import matplotlib.pyplot as plt

# Check versions
print(torch.__version__, torchvision.__version__)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path

from torch.utils.data import Dataset
from torchvision import transforms
from typing import Tuple, Dict, List # for typehints

In [ ]:
def find_classes(directory: str) -> Tuple[List[str], Dict[str, int]]:
  """Finds the class folder names in a target directory."""
  classes = sorted([entry.name for entry in list(os.scandir(directory)) if entry.is_dir()])

  if not classes:
    raise FileNotFoundError(f"Couldn't find any classes in directory... Please check file structure")

  class_to_idx = {class_name: i for i, class_name in enumerate(classes)}

  return classes, class_to_idx

In [ ]:
ROOT_DIR = ""
train_dir = Path(f"{ROOT_DIR}/train")
test_dir = Path(f"{ROOT_DIR}/test")

In [ ]:
from torch.utils.data import Dataset
from torch import Tensor

class FeatureMapFolder(Dataset):
  def __init__(self, targ_dir: str, transform=None):
    super().__init__()
    self.paths = list(Path(targ_dir).glob("*/*.pt"))
    self.transform = transform
    self.classes, self.class_to_idx = find_classes(targ_dir)
    
  def load_feature_map(self, index: int) -> Tensor:
    "Opens an image via a path and returns it"
    image_path = self.paths[index]
    return torch.load(image_path)
  
  def aggregate_channels(self, feature_map):    
    return torch.sum(feature_map, 0) / feature_map.shape[0]

  def __len__(self):
    "Returns the total number of samples"
    return len(self.paths)

  def __getitem__(self, index: int) -> Tuple[torch.Tensor, int]:
    "Returns one sample of data, data and label (X, y)"
    img = self.load_feature_map(index)
    img = img.squeeze()
    img = self.aggregate_channels(img)
    img = img.unsqueeze(0) 
    class_name = self.paths[index].parent.name 
    class_idx = self.class_to_idx[class_name]

    # Transform if necessary
    if self.transform:
      return self.transform(img), class_idx
    else:
      return img, class_idx


In [ ]:
from torchvision.transforms import transforms

train_transforms = transforms.Compose([
    transforms.ToTensor(),
])

test_transforms = transforms.Compose([
    transforms.ToTensor()
])

In [ ]:
# Test out image folder custom
train_data = FeatureMapFolder(train_dir)
test_data = FeatureMapFolder(test_dir)

In [ ]:
from torch.utils.data import DataLoader
BATCH_SIZE=16

train_dataloader = DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_dataloader = DataLoader(
    dataset=test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

img, label = next(iter(train_dataloader))

img.shape, label.shape 

In [ ]:
class Model_2(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=1), 
            nn.ReLU(),
            nn.BatchNorm2d(hidden_units),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout(0.3),
        )
        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2560, hidden_units),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(hidden_units, output_shape),
        )
    def forward(self, x):
        x = self.conv_block_1(x)
        x = self.layer_stack(x)
        return x

In [ ]:
model = Model_2(input_shape=1, hidden_units=10, output_shape=1)
model = model.to(device)

In [ ]:
model.load_state_dict(torch.load("./models/max_pool_layer_1_generalisability/model_3.pth"))

In [ ]:
from tqdm.auto import tqdm
import torchmetrics, mlxtend
print(f"mlxtend version: {mlxtend.__version__}")

In [ ]:
y_preds = []
target_preds = []
model.eval()
with torch.inference_mode():
    for X, y in tqdm(test_dataloader, desc="Making predictions"):
        X, y = X.to(device), y
        target_preds.append(y.cpu())

        y_logit = model(X)
        y_pred = torch.sigmoid(y_logit)
        y_labels = (y_pred > 0.5).float()
        y_preds.append(y_pred.cpu())

y_pred_tensor = torch.cat(y_preds).squeeze(1)
target_preds_tensor = torch.cat(target_preds)


In [ ]:
print(f"y_pred_tensor shape: {y_pred_tensor.shape}")
print(f"target_preds_tensor shape: {target_preds_tensor.shape}")

In [ ]:
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix

class_names = test_data.classes
print(f"Class names: {class_names}")

confmat = ConfusionMatrix(num_classes=2, task="binary") 
confmat_tensor = confmat(preds=y_pred_tensor, target=target_preds_tensor)

fig, ax = plot_confusion_matrix(conf_mat=confmat_tensor.numpy(), # matplotlib likes numpy
                 class_names=class_names,
                 figsize=(10, 7))

In [ ]:
from torchmetrics.classification import BinaryPrecision, BinaryRecall, BinaryAccuracy, BinaryAUROC

from tqdm.notebook import tqdm
def eval(model: torch.nn.Module, data_loader: torch.utils.data.DataLoader,
        loss_fn: torch.nn.Module, device=device):
  """Returns a dictionary containing the results of the model predicting on data_loader."""
  loss, acc, aurcroc_score, precision, recall = 0, 0, 0, 0, 0, 
  all_preds = []
  all_labels = []
  model.eval()
  model.to(device)
  torch.manual_seed(42)
  binary_precision_fn = BinaryPrecision().to(device)
  binary_recall_fn = BinaryRecall().to(device)
  binary_accuracy_fn = BinaryAccuracy().to(device)
  binary_aucroc_fn = BinaryAUROC().to(device)
  with torch.inference_mode():
    for X, y in tqdm(data_loader):
      X, y = X.to(device), y.to(device)

      y_pred = model(X).squeeze()
      loss += loss_fn(y_pred, y.float())
      acc += binary_accuracy_fn(torch.round(torch.sigmoid(y_pred)), y.float())

      all_preds.append(torch.sigmoid(y_pred))
      all_labels.append(y)

    loss /= len(data_loader)
    acc /= len(data_loader)
    
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    precision = binary_precision_fn(all_preds, all_labels)
    recall = binary_recall_fn(all_preds, all_labels)
    aurcroc_score += binary_aucroc_fn(all_preds, all_labels)
  return {
      "model_loss": loss.item(),
      "model_acc": acc.item(),
      "model_aucroc_score": aurcroc_score.item(),
      "model_precision": precision.item(),
      "model_recall": recall.item()}



In [ ]:
model_eval_results = eval(model=model,
    data_loader=test_dataloader,
    loss_fn=nn.BCEWithLogitsLoss(),
    device=device)

In [ ]:
import json
def saveModelAndResults(model: torch.nn.Module, results: Dict[str, List[float]], model_name: str):
    """Saves a model's eval results to disk"""
    results_save_path = f"./eval_results/max_pool_1/{model_name}_eval_results.json"
    if os.path.exists(results_save_path):
        print("Results file already exists. Terminating...")
        return
   
    with open(results_save_path, "w") as f:
        json.dump(results, f)
    print(f"Saved {model_name}!")

In [ ]:
print(model_eval_results)

In [ ]:
saveModelAndResults(model=model,
    results=model_eval_results,
    model_name="model_3")